### Scenario 2: A cross-functional team with one data scientist working on an ML model

MLflow setup:

Maybe managing mlow locally is fine in this case of situations, you could show the other members
teams the grogress and so on, but I also could storage the data in a remote server, like AWS S3, or google cloud storage.

- tracking server: yes, local server
- backend store: sqlite database
- artifacts store: local filesystem
The experiments can be explored locally by accessing the local tracking server.

To run this example you need to launch the mlflow server locally by running the following command in your terminal:

$ mlflow server --backend-store-uri sqlite:///backend.db

That gives us a uri to connect to the mlflow server

##### log runs

In [1]:
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score


mlflow.set_tracking_uri("http://127.0.0.1:5000")
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")
print(mlflow.search_experiments())

mlflow.set_experiment("experiment-01") 

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.3, "random_state": 41}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models", input_example=X[:5])
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")


tracking URI: 'http://127.0.0.1:5000'
[<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1744718324003, experiment_id='1', last_update_time=1744718324003, lifecycle_stage='active', name='experiment-01', tags={}>, <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1744717570395, experiment_id='0', last_update_time=1744717570395, lifecycle_stage='active', name='Default', tags={}>]
default artifacts URI: 'mlflow-artifacts:/1/cd3e003fb7f4457ab660e217de18e33b/artifacts'
🏃 View run adorable-stag-516 at: http://127.0.0.1:5000/#/experiments/1/runs/cd3e003fb7f4457ab660e217de18e33b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [14]:
# just in case we need to delete a run.
import mlflow
import shutil
import os

# remove all in the server 
mlflow.delete_run(run_id="0fcf0426e9d544d282b83ab675f9a1dd")

# remove the remain of artifacts in local
artifact_path = "mlartifacts/1/0fcf0426e9d544d282b83ab675f9a1dd"
# Delete the artifact directory if it exists
if os.path.exists(artifact_path):
    shutil.rmtree(artifact_path)
    print(f"Deleted artifacts at: {artifact_path}")
else:
    print(f"No artifacts found at: {artifact_path}")


Deleted artifacts at: mlartifacts/1/0fcf0426e9d544d282b83ab675f9a1dd


##### Register models

In [2]:
from mlflow.tracking import MlflowClient
from mlflow.exceptions import MlflowException

model_name = "iris-classifier"

client = MlflowClient(mlflow.get_tracking_uri())

try:
    client.get_registered_model(name=model_name)
except MlflowException:
    print("It's not possible to access the model registry :(")



It's not possible to access the model registry :(


In [8]:
run_ids = client.search_runs(experiment_ids=[1])
print(run_ids)

[<Run: data=<RunData: metrics={'accuracy': 0.9666666666666667}, params={'C': '0.3', 'random_state': '41'}, tags={'mlflow.log-model.history': '[{"run_id": "cd3e003fb7f4457ab660e217de18e33b", '
                             '"artifact_path": "models", "utc_time_created": '
                             '"2025-04-15 13:40:25.923050", "model_uuid": '
                             '"fe5ee5a444204e1ebea01fa998c9e4f8", "flavors": '
                             '{"python_function": {"model_path": "model.pkl", '
                             '"predict_fn": "predict", "loader_module": '
                             '"mlflow.sklearn", "python_version": "3.12.7", '
                             '"env": {"conda": "conda.yaml", "virtualenv": '
                             '"python_env.yaml"}}, "sklearn": '
                             '{"pickled_model": "model.pkl", '
                             '"sklearn_version": "1.6.1", '
                             '"serialization_format": "cloudpickle", "code": '

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
print(mlflow.search_experiments())

run_ids = ["59aa9e3e87894d698a67f70d2004f1af", ]

# we register the model
run_ids = ["59aa9e3e87894d698a67f70d2004f1af", "cd3e003fb7f4457ab660e217de18e33b"]

# every time this loop is executed it appears new versions
for run_id in run_ids:
    model_uri = f"runs:/{run_id}/models" # go to run and check if the path exists
    result = mlflow.register_model(
        model_uri=model_uri,
        name=model_name,
    )

Registered model 'iris-classifier' already exists. Creating a new version of this model...
2025/04/15 13:48:55 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 4
Created version '4' of model 'iris-classifier'.
Registered model 'iris-classifier' already exists. Creating a new version of this model...
2025/04/15 13:48:55 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 5


[<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1744718324003, experiment_id='1', last_update_time=1744718324003, lifecycle_stage='active', name='experiment-01', tags={}>]


Created version '5' of model 'iris-classifier'.


In [12]:
# we can remove the version non needed
client.delete_model_version(name="iris-classifier", version="3")


##### load models and check the accuracy

In [21]:
model_name = "iris-classifier"
mlflow.set_tracking_uri("http://127.0.0.1:5000")
client = MlflowClient(mlflow.get_tracking_uri())

model_ranking = {}
X, y = load_iris(return_X_y=True)
registered_model =client.get_registered_model(model_name)
for key, value in registered_model.aliases.items():
    print(f"Aliases: {key} -- Version: {value}")
    model_path = client.get_model_version_download_uri(name=model_name, version=value)
    model = mlflow.pyfunc.load_model(model_path)
    y_pred = model.predict(X)
    print(f"accuracy: {accuracy_score(y, y_pred)}")
    model_ranking[key] = {'model_name': model_name, 'version': value, 'accuracy': accuracy_score(y, y_pred)}

model_ranking

Aliases: bomba -- Version: 5


accuracy: 0.9666666666666667
Aliases: tontito -- Version: 4


accuracy: 0.96


{'bomba': {'model_name': 'iris-classifier',
  'version': '5',
  'accuracy': 0.9666666666666667},
 'tontito': {'model_name': 'iris-classifier',
  'version': '4',
  'accuracy': 0.96}}